In [3]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Gráfica comparativa por estación.

Para cada estación:
1. Se selecciona una semana completa a partir de WEEK_START.
2. Se representa la serie observada de O3 durante esa semana.
3. Se superponen, en el mismo gráfico, las predicciones de RF, XGBoost, GRU y LSTM
   únicamente para los últimos 3 días de esa semana.

Notas:
- RF y XGBoost usan la ventana de 72 horas sin normalización.
- GRU y LSTM usan scaler_X para la entrada y scaler_y para desescalar la salida.
- Si falta alguno de los modelos, la estación se omite para mantener la comparación completa.
"""

import os
import pickle
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model

# =============================================================================
# CONFIGURACIÓN
# =============================================================================

STATIONS = [
    "T1_E1_Alicante",
    "T1_E2_Elda",
    "T3_E1_Valencia",
    "T3_E2_Buñol",
    "T5_E1_Castellon",
    "T6_E2_Coratxa",
]

BASE_DIR = os.path.expanduser("/Volumes/copia_seguridad1/enviar_benja/carpeta sin título/clean/Finales/")

ENCODED_DIR = os.path.join(BASE_DIR, "encoded", "ml", "global")
OUTPUT_DIR = os.path.join(BASE_DIR, "comparative_forecasts")

TREE_MODEL_BASE = {
    "RF": os.path.join(BASE_DIR, "models", "random_forest", "global"),
    "XGBoost": os.path.join(BASE_DIR, "models", "xgboost", "global"),
}

RNN_MODEL_BASE = {
    "GRU": os.path.join(BASE_DIR, "models", "rnn", "global"),
    "LSTM": os.path.join(BASE_DIR, "models", "lstm", "global"),
}

SCALERS_BASE = os.path.join(BASE_DIR, "windows_partitioned", "global", "dl")

WINDOW_IN = 72
WINDOW_OUT = 72
WEEK_LENGTH_HOURS = 168
FORECAST_HOURS = 72

# Cambia esta fecha por la semana que quieras analizar
WEEK_START = pd.Timestamp("2024-05-01 00:00:00")

MODEL_STYLE = {
    "RF": {
        "color": "tab:red",
        "linestyle": "--",
        "linewidth": 2.0,
        "label": "RF",
    },
    "XGBoost": {
        "color": "tab:green",
        "linestyle": "-.",
        "linewidth": 2.0,
        "label": "XGBoost",
    },
    "GRU": {
        "color": "tab:orange",
        "linestyle": ":",
        "linewidth": 2.5,
        "label": "GRU",
    },
    "LSTM": {
        "color": "tab:purple",
        "linestyle": (0, (5, 2)),
        "linewidth": 2.0,
        "label": "LSTM",
    },
}

# =============================================================================
# FUNCIONES
# =============================================================================

def load_original_data(station: str) -> pd.DataFrame:
    """
    Carga el CSV de una estación, detecta la columna temporal y deja el índice
    en frecuencia horaria. Convierte los datos a numéricos e interpola huecos.
    """
    csv_path = os.path.join(ENCODED_DIR, f"{station}.csv")
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"No se encontró el archivo: {csv_path}")

    df = pd.read_csv(csv_path)

    time_col = None
    for col in df.columns:
        if col.lower() in {"timestamp", "datetime", "fecha", "time"}:
            time_col = col
            break
    if time_col is None:
        time_col = df.columns[0]

    df[time_col] = pd.to_datetime(df[time_col], errors="coerce")
    df = df.dropna(subset=[time_col]).copy()
    df = df.set_index(time_col).sort_index()

    df = df.asfreq("h")
    df = df.apply(pd.to_numeric, errors="coerce")
    df = df.interpolate(method="time", limit_direction="both")
    df = df.bfill().ffill()

    return df


def get_week_slice(df: pd.DataFrame, week_start: pd.Timestamp) -> pd.DataFrame:
    """
    Devuelve exactamente 168 horas desde week_start.
    """
    week_end = week_start + pd.Timedelta(hours=WEEK_LENGTH_HOURS - 1)
    week_df = df.loc[week_start:week_end].copy()

    if len(week_df) != WEEK_LENGTH_HOURS:
        raise ValueError(
            f"La semana seleccionada no contiene {WEEK_LENGTH_HOURS} horas exactas. "
            f"Se han obtenido {len(week_df)} filas."
        )

    return week_df


def get_input_window(df: pd.DataFrame, forecast_start: pd.Timestamp) -> pd.DataFrame:
    """
    Extrae las 72 horas previas al inicio del pronóstico.
    """
    start = forecast_start - pd.Timedelta(hours=WINDOW_IN)
    end = forecast_start - pd.Timedelta(hours=1)
    window_df = df.loc[start:end].copy()

    if len(window_df) != WINDOW_IN:
        raise ValueError(
            f"La ventana de entrada no contiene {WINDOW_IN} horas exactas. "
            f"Se han obtenido {len(window_df)} filas."
        )

    return window_df


def load_tree_model(model_kind: str, station: str):
    """
    Carga un modelo de tipo árbol serializado con pickle.
    """
    model_path = os.path.join(TREE_MODEL_BASE[model_kind], station, "model.pkl")
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"No se encontró el modelo: {model_path}")

    with open(model_path, "rb") as f:
        model = pickle.load(f)

    return model


def load_rnn_bundle(model_kind: str, station: str):
    """
    Carga un modelo Keras y sus scalers asociados.
    """
    model_path = os.path.join(RNN_MODEL_BASE[model_kind], station, "model.keras")
    scaler_X_path = os.path.join(SCALERS_BASE, station, "scaler_X.pkl")
    scaler_y_path = os.path.join(SCALERS_BASE, station, "scaler_y.pkl")

    if not os.path.exists(model_path):
        raise FileNotFoundError(f"No se encontró el modelo: {model_path}")
    if not os.path.exists(scaler_X_path):
        raise FileNotFoundError(f"No se encontró scaler_X: {scaler_X_path}")
    if not os.path.exists(scaler_y_path):
        raise FileNotFoundError(f"No se encontró scaler_y: {scaler_y_path}")

    model = load_model(model_path)

    with open(scaler_X_path, "rb") as f:
        scaler_X = pickle.load(f)

    with open(scaler_y_path, "rb") as f:
        scaler_y = pickle.load(f)

    return model, scaler_X, scaler_y


def predict_tree_model(model, input_window: pd.DataFrame) -> pd.Series:
    """
    Predicción para RF y XGBoost.
    """
    flat_input = input_window.values.reshape(1, -1)
    pred = model.predict(flat_input)

    if pred.ndim == 2:
        pred = pred.flatten()

    return pd.Series(pred[:WINDOW_OUT])


def predict_rnn_model(model, input_window: pd.DataFrame, scaler_X, scaler_y) -> pd.Series:
    """
    Predicción para GRU y LSTM.
    """
    X = scaler_X.transform(input_window.values)
    X = X.reshape(1, input_window.shape[0], input_window.shape[1])

    pred_norm = model.predict(X, verbose=0)

    if pred_norm.ndim == 2:
        pred_norm = pred_norm.flatten()

    pred_original = scaler_y.inverse_transform(pred_norm.reshape(-1, 1)).flatten()
    return pd.Series(pred_original[:WINDOW_OUT])


def plot_station_comparison(
    station: str,
    week_o3: pd.Series,
    predictions: dict,
    week_start: pd.Timestamp,
):
    """
    Genera una sola figura con la semana observada y las predicciones de los 4 modelos
    para los últimos 3 días.
    """
    forecast_start = week_start + pd.Timedelta(hours=WEEK_LENGTH_HOURS - FORECAST_HOURS)
    forecast_index = pd.date_range(start=forecast_start, periods=FORECAST_HOURS, freq="h")
    forecast_end = forecast_index[-1]

    fig, ax = plt.subplots(figsize=(15, 6))

    ax.plot(
        week_o3.index,
        week_o3.values,
        color="black",
        linewidth=1.8,
        label="O3 observado",
    )

    ax.axvspan(
        forecast_start,
        forecast_end,
        alpha=0.08,
        color="gray",
        label="Periodo de predicción",
    )

    for model_name in ["RF", "XGBoost", "GRU", "LSTM"]:
        pred = predictions[model_name]
        style = MODEL_STYLE[model_name]

        ax.plot(
            forecast_index,
            pred.values,
            color=style["color"],
            linestyle=style["linestyle"],
            linewidth=style["linewidth"],
            label=style["label"],
        )

    ax.axvline(
        forecast_start,
        color="dimgray",
        linestyle=":",
        linewidth=1.5,
    )

    ax.set_title(
        f"Serie semanal de O3 y predicciones de modelos\n"
        f"{station}  |  Semana: {week_start.strftime('%Y-%m-%d')}"
    )
    ax.set_xlabel("Fecha y hora")
    ax.set_ylabel("Concentración de O3")
    ax.grid(True, alpha=0.3)
    ax.legend(ncol=3)
    fig.tight_layout()

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    out_path = os.path.join(
        OUTPUT_DIR,
        f"{station}_week_{week_start.strftime('%Y%m%d')}_comparison.png",
    )
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)

    print(f"Gráfica guardada: {out_path}")


# =============================================================================
# PROGRAMA PRINCIPAL
# =============================================================================

def main():
    print("Generando gráficas comparativas por estación...")
    print(f"Directorio de salida: {OUTPUT_DIR}")
    print(f"Semana seleccionada: {WEEK_START}")

    for station in STATIONS:
        print(f"\nProcesando {station}")

        try:
            df = load_original_data(station)
        except Exception as e:
            print(f"  Error al cargar datos: {e}")
            continue

        if "O3" not in df.columns:
            print("  La columna O3 no existe en el CSV. Se omite la estación.")
            continue

        try:
            week_df = get_week_slice(df, WEEK_START)
        except Exception as e:
            print(f"  Error al extraer la semana: {e}")
            continue

        forecast_start = WEEK_START + pd.Timedelta(hours=WEEK_LENGTH_HOURS - FORECAST_HOURS)

        try:
            input_window = get_input_window(df, forecast_start)
        except Exception as e:
            print(f"  Error al extraer la ventana de entrada: {e}")
            continue

        predictions = {}

        try:
            rf_model = load_tree_model("RF", station)
            predictions["RF"] = predict_tree_model(rf_model, input_window)
        except Exception as e:
            print(f"  RF no disponible: {e}")

        try:
            xgb_model = load_tree_model("XGBoost", station)
            predictions["XGBoost"] = predict_tree_model(xgb_model, input_window)
        except Exception as e:
            print(f"  XGBoost no disponible: {e}")

        try:
            gru_model, gru_scaler_X, gru_scaler_y = load_rnn_bundle("GRU", station)
            predictions["GRU"] = predict_rnn_model(gru_model, input_window, gru_scaler_X, gru_scaler_y)
        except Exception as e:
            print(f"  GRU no disponible: {e}")

        try:
            lstm_model, lstm_scaler_X, lstm_scaler_y = load_rnn_bundle("LSTM", station)
            predictions["LSTM"] = predict_rnn_model(lstm_model, input_window, lstm_scaler_X, lstm_scaler_y)
        except Exception as e:
            print(f"  LSTM no disponible: {e}")

        if len(predictions) != 4:
            print("  No están disponibles los cuatro modelos. Se omite la estación.")
            continue

        try:
            plot_station_comparison(
                station=station,
                week_o3=week_df["O3"],
                predictions=predictions,
                week_start=WEEK_START,
            )
        except Exception as e:
            print(f"  Error al generar la gráfica: {e}")

    print("\nProceso completado.")


if __name__ == "__main__":
    main()

Generando gráficas comparativas por estación...
Directorio de salida: /Volumes/copia_seguridad1/enviar_benja/carpeta sin título/clean/Finales/comparative_forecasts
Semana seleccionada: 2024-05-01 00:00:00

Procesando T1_E1_Alicante
Gráfica guardada: /Volumes/copia_seguridad1/enviar_benja/carpeta sin título/clean/Finales/comparative_forecasts/T1_E1_Alicante_week_20240501_comparison.png

Procesando T1_E2_Elda
Gráfica guardada: /Volumes/copia_seguridad1/enviar_benja/carpeta sin título/clean/Finales/comparative_forecasts/T1_E2_Elda_week_20240501_comparison.png

Procesando T3_E1_Valencia
Gráfica guardada: /Volumes/copia_seguridad1/enviar_benja/carpeta sin título/clean/Finales/comparative_forecasts/T3_E1_Valencia_week_20240501_comparison.png

Procesando T3_E2_Buñol
Gráfica guardada: /Volumes/copia_seguridad1/enviar_benja/carpeta sin título/clean/Finales/comparative_forecasts/T3_E2_Buñol_week_20240501_comparison.png

Procesando T5_E1_Castellon
Gráfica guardada: /Volumes/copia_seguridad1/envia